In [464]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from datetime import datetime
import os
import requests
import pandas as pd
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.cluster import KMeans
from functools import reduce
import numpy as np
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, roc_curve, auc
import xgboost as xgb
import bson
import json
import gridfs
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose    import ColumnTransformer
import re
import joblib
import io


In [465]:
def json_serial(obj):
    """JSON serializer for objects not serializable by default."""
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

uri = os.getenv('SBS_V1_MONGO_URI')

client = MongoClient(uri, server_api=ServerApi('1'))
db = client['SBSV1']
fs = gridfs.GridFS(db)

try:
    client.admin.command('ping')
    print('Pinged your deployment. You successfully connected to MongoDB!')
except Exception as e:
    print(e)

nba_games_historical_collection = db['nba_games_historical']
nba_team_aggregated_game_stats_historical_collection = db['nba_team_aggregated_game_stats_historical']
nba_game_player_stats_historical_collection = db['nba_game_player_stats_historical']
nba_player_aggregated_game_stats_historical_collection = db['nba_player_aggregated_game_stats_historical']
nba_player_aggregated_odds_historical_collection = db['nba_player_aggregated_odds_historical']
cached_web_api_response_collection = db['cached_web_api_response']
ml_models_collection = db['ml_models']

Pinged your deployment. You successfully connected to MongoDB!


In [466]:
####################### CONSTS ##########################
nba_supported_bet_types = [
    'player_points',
    'player_assists',
    'player_rebounds',
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_threes',
]

In [467]:
############# SPORTS BETTING SANDBOX API ################

#########################################################
# get_events ############################################
def get_event(sports):
    url = 'https://sportsbettingsandboxapi.com/odds-api/events/get'
    response = requests.post(url, json={ 'sports': sports }).json()['data']
    return response
#########################################################

#########################################################
# get_events_odds ############################################
def get_event_odds(sports):
    url = 'https://sportsbettingsandboxapi.com/odds-api/event_odds/get'
    response = requests.post(url, json={ 'sports': sports }).json()['data']['events']
    return response
#########################################################

In [468]:
################### MONGO FUNCS #########################

#########################################################
# get_historical_nba_game_objs_from_season ##############
def get_historical_nba_game_objs_from_season(season):
    return list(nba_games_historical_collection.find({ 'season': season }))
#########################################################

#########################################################
# get_historical_nba_player_aggregated_game_stats_from_season
def get_historical_nba_player_aggregated_game_stats_from_season(season, season_type):
    return list(nba_player_aggregated_game_stats_historical_collection.find({ 'season': season, 'seasonType': season_type }))
#########################################################

#########################################################
# get_historical_nba_player_aggregated_game_stats_from_season
def get_nba_player_aggregated_odds_historical_from_season(season, season_type):
    return list(nba_player_aggregated_odds_historical_collection.find({ 'season': season, 'seasonType': season_type }))
#########################################################

In [469]:
################### HELPER FUNCS ########################

#########################################################
# transform_player_game_stats_to_df #####################
def transform_player_game_stats_objs_to_df(player_game_stats_objs):
    stats = [stat for player in player_game_stats_objs for stat in player['playerStats'].values()]
    return pd.DataFrame(stats)
#########################################################

#########################################################
# transform_player_odds_to_df ###########################
def transform_player_odds_to_df(player_odds_objs):
    full_rows = []
    for player_odds in player_odds_objs:
        player_id = player_odds['playerId']

        for game_id, odds_for_game in player_odds['playerOdds'].items():
            for bet_type, line in odds_for_game.items():
                if line and line['outcomes']:
                    try:
                        outcomes = line['outcomes']
                        over = list(filter(lambda x: x['name'] == 'Over', outcomes))[0]
                        under = list(filter(lambda x: x['name'] == 'Under', outcomes))[0]
                        json_obj = {
                            'over_odds': over['price'],
                            'over_line': over['point'],
                            'under_odds': under['price'],
                            'under_line': over['point'],
                            'bet_type': bet_type,
                            'gameId': int(game_id),
                            'playerId': int(player_id),
                            'line_time': line['lastUpdate']
                        }
    
                        full_rows.append(json_obj)
                    except Exception as e:
                        print(f'failed to parse outcomes: {outcomes}')

    df = pd.DataFrame(full_rows)
    return df
#########################################################

In [470]:
##################### ML FUNCS ##########################

#########################################################
# enrich_player_stats_df_with_player_role ###############
def enrich_player_stats_df_with_player_role(player_stats_df):
    player_stats_for_clustering = ['points', 'assists', 'totReb', 'fgm', 'fga', 'tpm', 'tpa', 'ftm', 'fta', 'turnovers', 'blocks', 'steals']
    
    player_stats_df['date'] = player_stats_df['dateStart'].str.split('T').str[0]
    unique_dates = sorted(player_stats_df['date'].unique())

    role_cluster_rows = []
    for date in unique_dates:
        # Filter out games with zero minutes and keep games before date
        df = player_stats_df[(player_stats_df['date'] < date) & (player_stats_df['min'] > 0)].copy()
       
        if df.empty:
            continue            
            
        # Normalize stats per 36 minutes
        for stat in player_stats_for_clustering:
            df[stat] = df[stat] * (36 / df['min'])
    
        # Group by playerId and calculate mean
        all_players_avg_stats = (
            df
            .sort_values('dateStart', ascending=False)
            .groupby('playerId')[player_stats_for_clustering]
            .mean()
            .reset_index()
        )
    
        all_players_avg_stats = all_players_avg_stats.dropna()
    
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(all_players_avg_stats.drop('playerId', axis=1))
    
        kmeans = KMeans(n_clusters=5, random_state=42)
        all_players_avg_stats['roleCluster'] = kmeans.fit_predict(X_scaled)
        all_players_avg_stats['date'] = date
        
        role_cluster_rows.append(all_players_avg_stats[['playerId', 'roleCluster', 'date']])

    # Combine all per-date role cluster snapshots
    full_role_cluster_df = pd.concat(role_cluster_rows, ignore_index=True)

    # Ensure numeric consistency in merge keys
    player_stats_df['playerId'] = player_stats_df['playerId'].astype('int64')
    full_role_cluster_df['playerId'] = full_role_cluster_df['playerId'].astype('int64')

    full_role_cluster_df['date'] = pd.to_datetime(full_role_cluster_df['date']).dt.tz_localize(None)
    full_role_cluster_df['date'] = pd.to_datetime(full_role_cluster_df['date'], unit='ms')
    
    player_stats_df['date'] = pd.to_datetime(player_stats_df['date']).dt.tz_localize(None)
    player_stats_df['date'] = pd.to_datetime(player_stats_df['date'], unit='ms')

    enriched_df = pd.merge_asof(
        player_stats_df.sort_values('date'),
        full_role_cluster_df.sort_values('date'),
        on='date',
        by=['playerId'],
        direction='backward'  # use the most recent past date
    )
    
    return enriched_df[enriched_df['roleCluster'].notna()]
#########################################################


#########################################################
# enrich_player_stats_df_with_dvp_stats #################
def enrich_player_stats_df_with_dvp_stats(player_stats_df):
    stats = ['points', 'assists', 'totReb', 'fgm', 'fga', 'tpm', 'tpa', 'turnovers', 'blocks', 'steals']
    rolling_windows = [10]

    # Extract date and ensure proper sorting
    player_stats_df['date'] = player_stats_df['dateStart'].str.split('T').str[0]
    player_stats_df = player_stats_df.sort_values(['opponentTeamId', 'roleCluster', 'date'])

    dvp_rows = []
    unique_dates = sorted(player_stats_df['date'].unique())

    for date in unique_dates:
        # Filter past games for each date
        df = player_stats_df[player_stats_df['date'] < date]
        if df.empty:
            continue

        rolling_dvp_for_date_rows = []

        # ROLLING AVG for each window
        for window in rolling_windows:
            dvp_team_cluster = (
                df.groupby(['opponentTeamId', 'roleCluster'])[stats]
                .apply(lambda x: x.rolling(window=window, min_periods=1).mean().iloc[-1])
                .reset_index()
            )
            dvp_team_cluster = dvp_team_cluster.rename(columns={stat: f'{window}_g_{stat}_team_dvp' for stat in stats})

            dvp_league_cluster = (
                df.groupby('roleCluster')[stats]
                .apply(lambda x: x.rolling(window=window, min_periods=1).mean().iloc[-1])
                .reset_index()
            )
            dvp_league_cluster = dvp_league_cluster.rename(columns={stat: f'{window}_g_{stat}_league_dvp' for stat in stats})

            # Merge to calculate pct_diff
            dvp_full = dvp_team_cluster.merge(dvp_league_cluster, on='roleCluster')

            for stat in stats:
                dvp_full[f'{window}_g_{stat}_dvp_pct_diff'] = (
                    (dvp_full[f'{window}_g_{stat}_team_dvp'] - dvp_full[f'{window}_g_{stat}_league_dvp']) /
                    dvp_full[f'{window}_g_{stat}_league_dvp']
                )

            rolling_dvp_for_date_rows.append(dvp_full)

        # EXPANDING AVG
        dvp_team_cluster = df.groupby(['opponentTeamId', 'roleCluster'])[stats].mean().reset_index()
        dvp_team_cluster = dvp_team_cluster.rename(columns={stat: f'all_g_{stat}_team_dvp' for stat in stats})

        dvp_league_cluster = df.groupby('roleCluster')[stats].mean().reset_index()
        dvp_league_cluster = dvp_league_cluster.rename(columns={stat: f'all_g_{stat}_league_dvp' for stat in stats})

        dvp_full = dvp_team_cluster.merge(dvp_league_cluster, on='roleCluster')

        for stat in stats:
            dvp_full[f'all_g_{stat}_dvp_pct_diff'] = (
                (dvp_full[f'all_g_{stat}_team_dvp'] - dvp_full[f'all_g_{stat}_league_dvp']) /
                dvp_full[f'all_g_{stat}_league_dvp']
            )

        rolling_dvp_for_date_rows.append(dvp_full)

        # Merge all for the date
        rolling_dvp_for_date_df = reduce(
            lambda left, right: pd.merge(left, right, on=['opponentTeamId', 'roleCluster']),
            rolling_dvp_for_date_rows
        )
        rolling_dvp_for_date_df['date'] = date
        dvp_rows.append(rolling_dvp_for_date_df)

    # Combine all per-date DvP snapshots
    full_dvp_df = pd.concat(dvp_rows, ignore_index=True)

    # Drop rows where keys are missing
    player_stats_df = player_stats_df.dropna(subset=['opponentTeamId', 'roleCluster', 'date'])
    full_dvp_df = full_dvp_df.dropna(subset=['opponentTeamId', 'roleCluster', 'date'])


    player_stats_df['date'] = pd.to_datetime(player_stats_df['date']).dt.tz_localize(None)
    player_stats_df['date'] = pd.to_datetime(player_stats_df['date'], unit='ms')
    
    full_dvp_df['date'] = pd.to_datetime(full_dvp_df['date']).dt.tz_localize(None)
    full_dvp_df['date'] = pd.to_datetime(full_dvp_df['date'], unit='ms')

    # Ensure numeric consistency in merge keys
    player_stats_df['opponentTeamId'] = player_stats_df['opponentTeamId'].astype('int64')
    player_stats_df['roleCluster'] = player_stats_df['roleCluster'].astype('int64')
    full_dvp_df['opponentTeamId'] = full_dvp_df['opponentTeamId'].astype('int64')
    full_dvp_df['roleCluster'] = full_dvp_df['roleCluster'].astype('int64')

    # Merge back
    enriched_df = pd.merge_asof(
        player_stats_df.sort_values('date'),
        full_dvp_df.sort_values('date'),
        on='date',
        by=['opponentTeamId', 'roleCluster'],
        direction='backward'
    )

    return enriched_df
#########################################################

#########################################################
# enrich_player_stats_df_with_rolling_stats #############
def enrich_player_stats_df_with_rolling_stats(player_stats_df):
    # Raw stats to be processed
    stats = ['points', 'assists', 'totReb', 'fgm', 'fga', 'tpm', 'tpa', 'turnovers', 'blocks', 'steals']
    rolling_windows = [5, 10, 20]

    # Extract date and ensure proper sorting
    player_stats_df['date'] = player_stats_df['dateStart'].str.split('T').str[0]
    player_stats_df = player_stats_df.sort_values(['playerId', 'date'])

    rolling_stats_rows = []
    unique_dates = sorted(player_stats_df['date'].unique())

    for date in unique_dates:
        # Filter past games for each date
        df = player_stats_df[player_stats_df['date'] < date]
        if df.empty:
            continue

        rolling_stats_for_date_rows = []

        # Calculate rolling averages and stds
        for window in rolling_windows:
            rolling_avg_df = (
                df.groupby('playerId')[stats]
                .apply(lambda x: x.rolling(window=window, min_periods=1).mean().iloc[-1])
                .reset_index()
            )
            rolling_avg_df = rolling_avg_df.rename(columns={stat: f'{window}_g_{stat}_roll_avg' for stat in stats})

            rolling_std_df = (
                df.groupby('playerId')[stats]
                .apply(lambda x: x.rolling(window=window, min_periods=2).std().iloc[-1])
                .reset_index()
            )
            rolling_std_df = rolling_std_df.rename(columns={stat: f'{window}_g_{stat}_roll_std' for stat in stats})

            rolling_stats_for_date_rows.extend([rolling_avg_df, rolling_std_df])

        # Expanding average (all games up to the current date)
        expanding_avg_df = (
            df.groupby('playerId')[stats]
            .mean()
            .reset_index()
        ).rename(columns={stat: f'all_g_{stat}_roll_avg' for stat in stats})

        rolling_stats_for_date_rows.append(expanding_avg_df)

        # Merge all rolling stats for the current date
        rolling_stats_for_date_df = reduce(
            lambda left, right: pd.merge(left, right, on='playerId'), 
            rolling_stats_for_date_rows
        )
        rolling_stats_for_date_df['date'] = date
        rolling_stats_rows.append(rolling_stats_for_date_df)

    # Combine all per-date rolling stats
    full_rolling_stats_df = pd.concat(rolling_stats_rows, ignore_index=True)

    # Z-Score Calculation relative to the 20-game rolling average
    for window in rolling_windows[:len(rolling_windows)-1]:  # Only 5 and 10 game windows use 20-game avg for Z-score
        for stat in stats:
            z_score_col = f'{window}_g_{stat}_z_score'
            avg_col = f'{window}_g_{stat}_roll_avg'
            std_col = f'{window}_g_{stat}_roll_std'
            reference_avg_col = f'{rolling_windows[len(rolling_windows)-1]}_g_{stat}_roll_avg'

            # Ensure the 20-game rolling avg exists for each row
            if reference_avg_col in full_rolling_stats_df.columns:
                full_rolling_stats_df[z_score_col] = (
                    (full_rolling_stats_df[avg_col] - full_rolling_stats_df[reference_avg_col]) /
                    full_rolling_stats_df[std_col]
                )
                # Handle division by zero or NaN values
                full_rolling_stats_df[z_score_col] = full_rolling_stats_df[z_score_col].fillna(0)


    player_stats_df['date'] = pd.to_datetime(player_stats_df['date']).dt.tz_localize(None)
    player_stats_df['date'] = pd.to_datetime(player_stats_df['date'], unit='ms')
    
    full_rolling_stats_df['date'] = pd.to_datetime(full_rolling_stats_df['date']).dt.tz_localize(None)
    full_rolling_stats_df['date'] = pd.to_datetime(full_rolling_stats_df['date'], unit='ms')
    
    player_stats_df['playerId'] = player_stats_df['playerId'].astype('int64')
    full_rolling_stats_df['playerId'] = full_rolling_stats_df['playerId'].astype('int64')

    # Merge back with the original DataFrame
    enriched_df = pd.merge_asof(
        player_stats_df.sort_values('date'),
        full_rolling_stats_df.sort_values('date'),
        on='date',
        by='playerId',
        direction='backward'
    )


    return enriched_df
#########################################################

In [471]:
################# ML PIPELINE FUNCS #####################
def enrich_feature_map_with_analytics(player_stats_df):
    print('Enriching feature map with role clustering')
    enriched_player_stats_df = enrich_player_stats_df_with_player_role(player_stats_df)
    print('Enriching feature map with DVP stats')
    enriched_player_stats_df = enrich_player_stats_df_with_dvp_stats(enriched_player_stats_df)
    print('Enriching feature map with rolling stats, std, and z-score stats')
    enriched_player_stats_df = enrich_player_stats_df_with_rolling_stats(enriched_player_stats_df)
    return enriched_player_stats_df

def enrich_feature_map_with_odds_data_for_bet_type(player_stats_df, player_odds_df, bet_type):
    filtered_player_odds_by_bet_type = player_odds_df[player_odds_df['bet_type'] == bet_type]
    return pd.merge(filtered_player_odds_by_bet_type, player_stats_df, on=['gameId', 'playerId'])


def time_based_target_encode_smoothing(df, col, target, date_col, smoothing=10):
    # Ensure chronological sorting
    df = df.sort_values(by=date_col)
    global_mean = df[target].mean()

    encoded_col = np.zeros(len(df))
    encoding_mapping = {}

    for i in range(len(df)):
        # Use only past data to encode the current row
        train_data = df.iloc[:i]

        if len(train_data) == 0:
            encoded_value = global_mean
        else:
            agg = train_data.groupby(col)[target].agg(['mean', 'count'])
            smooth = (agg['mean'] * agg['count'] + global_mean * smoothing) / (agg['count'] + smoothing)
            encoded_value = smooth.get(df.iloc[i][col], global_mean)
        
        encoded_col[i] = encoded_value
        
        # Store the mapping for later use in inference
        key = (df.iloc[i][col], target)
        encoding_mapping[key] = encoded_value

    return encoded_col, encoding_mapping

def apply_time_based_target_encoding(df, categorical_features, target_cols, date_col, smoothing=10):
    encoding_mappings = {}
    for col in categorical_features:
        for target in target_cols:
            encoded_col_name = f'{col}_{target}_time_enc'
            encoded_col, mapping = time_based_target_encode_smoothing(df, col, target, date_col, smoothing)
            df[encoded_col_name] = encoded_col
            # Save mapping with feature name and target
            encoding_mappings[(col, target)] = mapping

    return df, encoding_mappings


def remove_highly_correlated_features(df, correlation_threshold=0.9):
    """
    Removes features with a correlation higher than the specified threshold.

    Args:
        df (pd.DataFrame): The input dataframe.
        correlation_threshold (float): The correlation threshold to identify features to drop.

    Returns:
        pd.DataFrame: The dataframe with highly correlated features removed.
    """
    # Compute the correlation matrix
    corr_matrix = df.corr().abs()

    # Select the upper triangle of the correlation matrix
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Identify features to drop
    to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > correlation_threshold)]

    # Drop the features
    df_reduced = df.drop(columns=to_drop)
    
    return df_reduced

# Example usage:
# feature_map_df = <your_normalized_feature_map_dataframe>
# feature_map_df = remove_highly_correlated_features(feature_map_df)

def apply_pca(feature_map_df, n_components=None):
    # Extract numeric columns
    numeric_cols = feature_map_df.select_dtypes(include=['float64', 'int64']).columns
    X = feature_map_df[numeric_cols]

    # Apply PCA
    pca = PCA(n_components=n_components)
    pca_transformed = pca.fit_transform(X)

    # Create a DataFrame for the PCA components
    pca_columns = [f'PC{i+1}' for i in range(pca_transformed.shape[1])]
    pca_df = pd.DataFrame(pca_transformed, columns=pca_columns)

    # Explained variance ratio plot
    plt.figure(figsize=(10, 6))
    plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance')
    plt.title('PCA - Cumulative Explained Variance')
    plt.grid()
    plt.show()

    # Return the transformed data and PCA object for further inspection
    return pca_df, pca

def process_nba_player_training_data_for_model(feature_map_df, bet_type):
    # remove rows where line is not there
    feature_map_df = feature_map_df[feature_map_df['over_line'].notna()]
    feature_map_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    feature_map_df.fillna(0, inplace=True)

    # creating target variables
    raw_stat = None
    if bet_type == 'player_points':
        raw_stat = 'points'
    if bet_type == 'player_assists':
        raw_stat = 'assists'
    if bet_type == 'player_rebounds':
        raw_stat = 'totReb'
    if bet_type == 'player_threes':
        raw_stat = 'tpm'
    if bet_type == 'player_points_rebounds_assists':
        feature_map_df['PRA'] = feature_map_df['points'] + feature_map_df['totReb'] + feature_map_df['assists']
        raw_stat = 'PRA'
    if bet_type == 'player_points_rebounds':
        feature_map_df['PR'] = feature_map_df['points'] + feature_map_df['totReb']
        raw_stat = 'PR'
    if bet_type == 'player_points_assists':
        feature_map_df['PA'] = feature_map_df['points'] + feature_map_df['assists']
        raw_stat = 'PA'

    feature_map_df['prob_over'] = (feature_map_df[raw_stat] > feature_map_df['over_line']).astype(int)
    feature_map_df['prob_under'] = 1 - feature_map_df['prob_over']

    # your list of substrings
    # used to get rid of features that might not correlate to what we want to predict
    stats_for_feature_engineering = ['points', 'assists', 'totReb', 'fgm', 'fga', 'tpm', 'tpa', 'turnovers', 'blocks', 'steals']
    words = ['points', 'assists', 'totReb', 'fgm', 'fga',
         'tpm', 'tpa', 'turnovers', 'blocks', 'steals']
    filtered_words = []
    if raw_stat in stats_for_feature_engineering:
        filtered_words = list(filter(lambda x: x != raw_stat, words))
    else:
        if raw_stat == 'PRA':
            stats_to_keep = ['points', 'assists', 'totReb']
            filtered_words = list(filter(lambda x: x not in stats_to_keep, words))
        if raw_stat == 'PR':
            stats_to_keep = ['points', 'totReb']
            filtered_words = list(filter(lambda x: x not in stats_to_keep, words))
        if raw_stat == 'PA':
            stats_to_keep = ['points', 'assists']
            filtered_words = list(filter(lambda x: x not in stats_to_keep, words))

    # build an “or” regex, escaping just in case
    pattern = r'(' + r'|'.join(map(re.escape, filtered_words)) + r')'
    to_drop = feature_map_df.filter(regex=pattern, axis=1).columns
    feature_map_df = feature_map_df.drop(columns=to_drop)

    # removing cols
    cols_to_drop = [
        "points", "assists", "totReb", "fgm", "fga", "fgp", "ftm", "fta", "ftp", 
        "tpm", "tpa", "tpp", "offReb", "defReb", "plusMinus", "pFouls", "steals", 
        "turnovers", "blocks", "gameId", "line_time", "dateStart", "over_line", 
        "under_line", "min", "win"
    ]

    for c in cols_to_drop:
        if c in feature_map_df.columns.tolist():
            feature_map_df.drop(c, axis=1, inplace=True)
            
    print(f'for bet_type = {bet_type}, feature_map = ', feature_map_df.tail(1).to_json(orient='records', indent=2))

    # normalization 
    categorical_features = [
        'playerId', 
        'teamId',
        'bet_type',
        'opponentTeamId', 
        'date', 
        'season', 
        'season_type',
        'roleCluster',
        'isHome'
    ]

    date_col = 'date'

    target_cols = ['prob_over', 'prob_under']
    
    # get numeric columns
    numeric_cols = feature_map_df.select_dtypes(include=['float64', 'int64']).columns
    numeric_cols = [col for col in numeric_cols if col not in categorical_features and col not in target_cols]
    # print('full feature map', feature_map_df.tail(1).to_json(orient='records', indent=2))
    # print('numeric cols only', feature_map_df[numeric_cols].tail(1).to_json(orient='records', indent=2))
    print("Checking for NaN values:\n", feature_map_df[numeric_cols].isna().sum())
    print("Checking for Inf values:\n", feature_map_df[numeric_cols].applymap(np.isinf).sum())

    # sort by date
    feature_map_df = feature_map_df.sort_values(by=date_col)
    
    # extract date col and target cols
    date_col_df = feature_map_df[date_col]
    target_cols_df = feature_map_df[target_cols]

    # Initialize the scaler
    scaler = StandardScaler()
    
    # Apply the scaler
    feature_map_df[numeric_cols] = scaler.fit_transform(feature_map_df[numeric_cols])

    scaler_params = {
        "bet_type":   bet_type,
        "with_mean":  scaler.with_mean,
        "with_std":   scaler.with_std,
        "mean":       scaler.mean_.tolist(),
        "scale":      scaler.scale_.tolist(),
        "var":        scaler.var_.tolist(),
        "fitted_at":  datetime.utcnow().isoformat()
    }

    # target encode categorical features
    # feature_map_df, encoding_mappings = apply_time_based_target_encoding(feature_map_df, categorical_features, target_cols, date_col)
    
    # drop the categorical features now that they are encoded
    # feature_map_df.drop(categorical_features, axis=1, inplace=True)
    # drop the target cols 
    # feature_map_df.drop(target_cols, axis=1, inplace=True)
    
    # remove highly correlated features
    feature_map_df_numeric_cols = remove_highly_correlated_features(feature_map_df[numeric_cols])
    feature_map_df = feature_map_df_numeric_cols.join(feature_map_df[categorical_features])
    
    # apply pca 
    # pca_df, pca_model = apply_pca(feature_map_df, 72)

    feature_map_df['date'] = feature_map_df['date'].apply(lambda dt: dt.toordinal())
    
    return feature_map_df, date_col_df, target_cols_df, scaler_params


def get_raw_training_data_for_sbs_nba_ensemble_model_1_mapped_by_bet_type(season, season_type):
    print(f'getting player data for season: {season}, season_type: {season_type}')
    player_stats_objs = get_historical_nba_player_aggregated_game_stats_from_season(season, season_type)
    player_stats_df = transform_player_game_stats_objs_to_df(player_stats_objs)
    enriched_player_stats_df = enrich_feature_map_with_analytics(player_stats_df)
    
    print(f'getting player odds data for season: {season}, season_type: {season_type}')
    player_odds_objs = get_nba_player_aggregated_odds_historical_from_season(season, season_type)
    player_odds_df = transform_player_odds_to_df(player_odds_objs)
    
    feature_maps_by_bet_types = dict()
    for bet_type in nba_supported_bet_types:
        full_feature_map = enrich_feature_map_with_odds_data_for_bet_type(enriched_player_stats_df, player_odds_df, bet_type)
        full_feature_map['season'] = season
        full_feature_map['season_type'] = season_type        
        feature_maps_by_bet_types[bet_type] = full_feature_map
    return feature_maps_by_bet_types
#########################################################

In [472]:
def get_training_date_and_target_dfs_for_sbs_ensemble_model_1():
    today = datetime.today()
    today_str = today.strftime('%Y-%m-%d')

    model_mongo_docs = list(ml_models_collection.find({ '_id': f'sbs_nba_ensemble_model_1_{today_str}' }))
    sbs_ensemble_model_raw_training_data_full = dict()

    # Fetch from GridFS
    if len(model_mongo_docs) > 0:
        model_mongo_doc = model_mongo_docs[0]

        for bet_type in nba_supported_bet_types:
            file_id = model_mongo_doc.get(f'sbs_ensemble_model_raw_training_data_full_{bet_type}')
            print(f"Fetching {bet_type} data with file_id: {file_id}")

            if file_id:
                try:
                    with fs.get(file_id) as file_data:
                        raw_data = file_data.read()
                        print(f"Retrieved {bet_type} data size: {len(raw_data)} bytes")

                        json_data = json.loads(raw_data.decode('utf-8'))
                        sbs_ensemble_model_raw_training_data_full[bet_type] = pd.DataFrame(json_data)

                except Exception as e:
                    print(f"Error reading {bet_type} data from GridFS: {e}")
                    sbs_ensemble_model_raw_training_data_full[bet_type] = None
            else:
                print(f"No file found for {bet_type}")
                sbs_ensemble_model_raw_training_data_full[bet_type] = None
    else:
        sbs_ensemble_model_raw_training_data_2024 = get_raw_training_data_for_sbs_nba_ensemble_model_1_mapped_by_bet_type(2024, 'ALL')
        sbs_ensemble_model_raw_training_data_2023 = get_raw_training_data_for_sbs_nba_ensemble_model_1_mapped_by_bet_type(2023, 'ALL')
        
        latest_raw_training_data = sbs_ensemble_model_raw_training_data_2024
        all_historical_raw_training_data = [sbs_ensemble_model_raw_training_data_2023]
        
        sbs_ensemble_model_raw_training_data_full = {}
        for bet_type, df in latest_raw_training_data.items():
            all_dfs_for_bet_type = [df] + [historical_raw_training_data[bet_type] for historical_raw_training_data in all_historical_raw_training_data]
            combined_df = pd.concat(all_dfs_for_bet_type, ignore_index=True).sort_values(by='date')
            sbs_ensemble_model_raw_training_data_full[bet_type] = combined_df

            # Serialize and store in GridFS
            json_data = json.dumps(combined_df.to_dict(orient="records"), default=json_serial)
            file_id = fs.put(json_data.encode('utf-8'))

            # Update MongoDB with the GridFS reference
            ml_models_collection.update_one(
                { '_id': f'sbs_nba_ensemble_model_1_{today_str}' },
                { "$set": { f'sbs_ensemble_model_raw_training_data_full_{bet_type}': file_id } },
                upsert=True
            )

    # Process the data and store processed versions
    all_training_data_dfs = {}
    for bet_type in nba_supported_bet_types:
        processed_feature_map, date_col, target_cols, scaler_params = None, None, None, None

        if len(model_mongo_docs) > 0:
            model_mongo_doc = model_mongo_docs[0]
            file_id = model_mongo_doc.get(f'sbs_ensemble_model_processed_training_data_{bet_type}')
            print(f"Fetching processed {bet_type} data with file_id: {file_id}")

            if file_id:
                try:
                    with fs.get(file_id) as file_data:
                        raw_data = file_data.read()
                        print(f"Retrieved processed {bet_type} data size: {len(raw_data)} bytes")

                        data = json.loads(raw_data.decode('utf-8'))
                        print(f"Decoded JSON keys for {bet_type}: {data.keys()}")

                        processed_feature_map = pd.DataFrame(data['processed_feature_map'])
                        date_col = pd.Series(data['date_col'])
                        target_cols = pd.DataFrame(data['target_cols'])

                except Exception as e:
                    print(f"Error reading processed {bet_type} data from GridFS: {e}")
                    processed_feature_map, date_col, target_cols, scaler_params = None, None, None, None
            else:
                print(f"No processed file found for {bet_type}")

        # If data is not retrieved, process the raw data
        if processed_feature_map is None or date_col is None or target_cols is None:
            print(f"Processing raw training data for {bet_type}")
            if bet_type in sbs_ensemble_model_raw_training_data_full and sbs_ensemble_model_raw_training_data_full[bet_type] is not None:
                try:
                    processed_feature_map, date_col, target_cols, scaler_params = process_nba_player_training_data_for_model(
                        sbs_ensemble_model_raw_training_data_full[bet_type], bet_type
                    )
                    
                    # Serialize and store in GridFS
                    json_data = json.dumps({
                        'processed_feature_map': processed_feature_map.to_dict(orient="records"),
                        'date_col': date_col.to_list(),
                        'target_cols': target_cols.to_dict(orient="records"),
                        'scaler_params': scaler_params
                    }, default=json_serial)

                    file_id = fs.put(json_data.encode('utf-8'))

                    # Update MongoDB with the GridFS reference
                    ml_models_collection.update_one(
                        { '_id': f'sbs_nba_ensemble_model_1_{today_str}' },
                        { "$set": { f'sbs_ensemble_model_processed_training_data_{bet_type}': file_id } },
                        upsert=True
                    )

                except Exception as e:
                    print(f"Error processing raw data for {bet_type}: {e}")
                    processed_feature_map, date_col, target_cols, scaler_params = None, None, None, None

        all_training_data_dfs[bet_type] = {
            'processed_feature_map': processed_feature_map,
            'date_col': date_col,
            'target_cols': target_cols,
            'scaler_params': scaler_params
        }

    return all_training_data_dfs


In [473]:
training_data_by_bet_type = get_training_date_and_target_dfs_for_sbs_ensemble_model_1()

Fetching player_points data with file_id: 681f6dd8b2d32923dc220e52
Retrieved player_points data size: 192092585 bytes
Fetching player_assists data with file_id: 681f6e18b2d32923dc221133
Retrieved player_assists data size: 125635180 bytes
Fetching player_rebounds data with file_id: 681f6e4eb2d32923dc221316
Retrieved player_rebounds data size: 191130158 bytes
Fetching player_points_rebounds_assists data with file_id: 681f6e7db2d32923dc2215f3
Retrieved player_points_rebounds_assists data size: 140221121 bytes
Fetching player_points_rebounds data with file_id: 681f6e9fb2d32923dc22180d
0:	test: 0.5082206	best: 0.5082206 (0)	total: 235ms	remaining: 46.8s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5278175562
bestIteration = 13

Shrink model to first 14 iterations.
0:	test: 0.5082206	best: 0.5082206 (0)	total: 3.39ms	remaining: 675ms
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5296578114
bestIteration = 41

Shrink model to first 42 iterations.
0

In [474]:
# TRAINING

def train_test_split_percentage(X, dates, targets, split_percentage):
    """ 
    Split data into training and testing sets based on percentage.
    """
    split_index = int(len(dates) * split_percentage)
    
    X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
    y_train, y_test = targets.iloc[:split_index], targets.iloc[split_index:]
    dates_train, dates_test = dates.iloc[:split_index], dates.iloc[split_index:]

    return X_train, X_test, y_train, y_test, dates_train, dates_test


def train_logistic_regression(X_train, y_train):
    """ 
    Train two separate Logistic Regression models:
    - One for prob_over
    - One for prob_under
    """
    model_over = LogisticRegression(max_iter=1000, random_state=42)
    model_under = LogisticRegression(max_iter=1000, random_state=42)

    model_over.fit(X_train, y_train['prob_over'])
    model_under.fit(X_train, y_train['prob_under'])

    return model_over, model_under


def evaluate_model(model_over, model_under, X_test, y_test):
    """ Evaluate model and plot ROC curve """
    # Predict probabilities
    y_pred_probs_over = model_over.predict_proba(X_test)[:, 1]
    y_pred_probs_under = model_under.predict_proba(X_test)[:, 1]

    # Calculate AUC-ROC for prob_over
    auc_score_over = roc_auc_score(y_test['prob_over'], y_pred_probs_over)
    auc_score_under = roc_auc_score(y_test['prob_under'], y_pred_probs_under)

    print(f"AUC-ROC Score - Over: {auc_score_over}")
    print(f"AUC-ROC Score - Under: {auc_score_under}")

    # Classification Reports
    print("\nClassification Report - Over:\n", classification_report(y_test['prob_over'], (y_pred_probs_over >= 0.5).astype(int)))
    print("\nClassification Report - Under:\n", classification_report(y_test['prob_under'], (y_pred_probs_under >= 0.5).astype(int)))

    # Plot ROC curve
    fpr_over, tpr_over, _ = roc_curve(y_test['prob_over'], y_pred_probs_over)
    fpr_under, tpr_under, _ = roc_curve(y_test['prob_under'], y_pred_probs_under)

    plt.figure(figsize=(10, 6))
    plt.plot(fpr_over, tpr_over, label=f"Prob Over (AUC = {auc_score_over:.2f})")
    plt.plot(fpr_under, tpr_under, label=f"Prob Under (AUC = {auc_score_under:.2f})")
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def run_logistic_regression_pipeline(X, dates, targets, split_percentage):
    """ Execute the pipeline with a percentage-based split """
    # Split data
    X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split_percentage(X, dates, targets, split_percentage)
    
    # Train models
    model_over, model_under = train_logistic_regression(X_train, y_train)
    
    # Evaluate models
    evaluate_model(model_over, model_under, X_test, y_test)

    return model_over, model_under, dates_test


# Example usage
# Assuming X, dates, targets are the dataframes you provided
# split_percentage = 0.8  # 80% training, 20% testing
# model_over, model_under, dates_test = run_logistic_regression_pipeline(X, dates, targets, split_percentage)
def train_xgboost(X, dates, targets, split_percentage):
    # Split data
    X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split_percentage(X, dates, targets, split_percentage)

    # Define the model
    xgb_over = XGBClassifier(
    max_depth=5, 
    learning_rate=0.1, 
    n_estimators=200, 
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss")
    
    xgb_under = XGBClassifier(
    max_depth=5, 
    learning_rate=0.1, 
    n_estimators=200, 
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss")

    # Train models for over and under
    xgb_over.fit(X_train, y_train['prob_over'])
    xgb_under.fit(X_train, y_train['prob_under'])

    # Predict
    prob_over_preds = xgb_over.predict(X_test)
    prob_under_preds = xgb_under.predict(X_test)

    # Evaluate using AUC-ROC
    auc_over = roc_auc_score(y_test['prob_over'], prob_over_preds)
    auc_under = roc_auc_score(y_test['prob_under'], prob_under_preds)

    print(f"AUC-ROC Score - Over: {auc_over}")
    print(f"AUC-ROC Score - Under: {auc_under}")

    # Classification Report
    print("\nClassification Report - Over:")
    print(classification_report(y_test['prob_over'], (prob_over_preds > 0.5).astype(int)))

    print("\nClassification Report - Under:")
    print(classification_report(y_test['prob_under'], (prob_under_preds > 0.5).astype(int)))

    return xgb_over, xgb_under, dates_test, prob_over_preds, prob_under_preds

# Example usage:
# models = train_xgboost(X_processed, dates, y_targets, 0.8)

def build_and_train_pipelines(
    X, dates, targets,
    categorical_features,
    split_pct: float = 0.8,
    calib_pct: float = 0.2,
    param_grid: dict = None,
    cv_splits: int = 3,
    n_iter: int = 10,
    calibration_method: str = 'isotonic',
    early_stopping_rounds: int = 50,
    random_state: int = 42,
    verbose: int = 100
):
    """
    Returns two fitted pipelines (over/under) and a dict of test AUCs.
    """
    # 1) Chronological train/test split
    split_idx = int(len(dates) * split_pct)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = targets.iloc[:split_idx], targets.iloc[split_idx:]

    # 1b) carve off a slice of train for calibration + early stopping
    calib_split = int(len(X_train) * (1 - calib_pct))
    X_base, X_calib = X_train.iloc[:calib_split], X_train.iloc[calib_split:]
    y_base, y_calib = y_train.iloc[:calib_split], y_train.iloc[calib_split:]

    # 2) default grid if none provided
    if param_grid is None:
        param_grid = {
            'clf__iterations':    [200, 400, 800],
            'clf__learning_rate': [0.001, 0.01, 0.1],
            'clf__depth':         [4, 6, 8],
            'clf__l2_leaf_reg':   [1, 3, 5],
        }

    def _make_pipeline(target_col):
        # CatBoost step
        cat = CatBoostClassifier(
            random_seed=random_state,
            verbose=verbose,
            eval_metric='AUC',
            early_stopping_rounds=early_stopping_rounds,
            use_best_model=True
        )
        pipe = Pipeline([('clf', cat)])

        # RandomizedSearchCV over that pipeline
        tscv = TimeSeriesSplit(n_splits=cv_splits)
        search = RandomizedSearchCV(
            pipe,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=tscv,
            scoring='roc_auc',
            n_jobs=-1,
            random_state=random_state,
            verbose=1
        )

        # fit with eval_set for early stopping
        search.fit(
            X_base, y_base[target_col],
            clf__cat_features=categorical_features,
            clf__eval_set=[(X_calib, y_calib[target_col])]
        )
        print(f"{target_col} → best_params = {search.best_params_}")

        # single-fold calibration (notice: estimator as positional arg)
        calibrator = CalibratedClassifierCV(
            search.best_estimator_,
            method=calibration_method,
            cv='prefit'
        )
        calibrator.fit(X_calib, y_calib[target_col])

        # wrap it in a “dummy” pipeline so you can treat it uniformly
        return Pipeline([('calib', calibrator)])

    # build & train both pipelines
    pipeline_over  = _make_pipeline('prob_over')
    pipeline_under = _make_pipeline('prob_under')

    # evaluate on the hold-out test
    y_pred_over  = pipeline_over.predict_proba(X_test)[:, 1]
    y_pred_under = pipeline_under.predict_proba(X_test)[:, 1]
    auc_over  = roc_auc_score(y_test['prob_over'],  y_pred_over)
    auc_under = roc_auc_score(y_test['prob_under'], y_pred_under)

    print(f"Final Over AUC:  {auc_over:.4f}")
    print(f"Final Under AUC: {auc_under:.4f}")
    print("\nOver Classification Report:")
    print(classification_report(
        y_test['prob_over'], (y_pred_over > .5).astype(int), digits=4
    ))
    print("\nUnder Classification Report:")
    print(classification_report(
        y_test['prob_under'], (y_pred_under > .5).astype(int), digits=4
    ))

    return pipeline_over, pipeline_under, {'auc_over': auc_over, 'auc_under': auc_under}

In [475]:
def cache_sbs_nba_ensemble_model_1_trained_pipelines(training_data_by_bet_type):
    today = datetime.today()
    today_str = today.strftime('%Y-%m-%d')

    categorical_features = [
        'playerId', 
        'teamId',
        'bet_type',
        'opponentTeamId', 
        'date', 
        'season', 
        'season_type',
        'roleCluster',
        'isHome'
    ]
    
    results = []

    for bet_type, info in training_data_by_bet_type.items():
        processed_feature_map = info.get('processed_feature_map')
        date_col = info.get('date_col')
        target_cols = info.get('target_cols')
        scaler_params = info.get('scaler_params') 

        # guard missing data
        if (processed_feature_map is None or processed_feature_map.shape[0] == 0
          or date_col is None or len(date_col) == 0
          or target_cols is None or len(target_cols) == 0):
            print(f"⚠️  Skipping '{bet_type}' — missing data.")
            continue

        print(f"\n=== TRAINING {bet_type} ===")
        try:
            pipe_over, pipe_under, metrics = build_and_train_pipelines(
                X=processed_feature_map,
                dates=date_col,
                targets=target_cols,
                categorical_features=categorical_features,
                split_pct=0.8,
                calib_pct=0.2,
                cv_splits=3,
                n_iter=10,
                calibration_method='isotonic',
                early_stopping_rounds=50,
                random_state=42,
                verbose=100
            )
        except Exception as e:
            print(f"❌  Failed on '{bet_type}': {e}\nSkipping.")
            continue

        # your stored scaler_params & pipelines
        print(scaler_params)
        scaler = StandardScaler(
            with_mean=scaler_params["with_mean"],
            with_std=scaler_params["with_std"]
        )
        scaler.mean_  = np.array(scaler_params["mean"])
        scaler.scale_ = np.array(scaler_params["scale"])
        scaler.var_   = np.array(scaler_params["var"])
        
        # keep the exact training columns
        feature_cols = processed_feature_map.columns.tolist()
        
        # derive numeric features as “all minus cats”
        numeric_features = [c for c in feature_cols if c not in categorical_features]
        
        # 1) align incoming X to the same columns
        align_cols = FunctionTransformer(
            lambda df: df.reindex(columns=feature_cols, fill_value=0),
            validate=False
        )
        
        # 2) ColumnTransformer: scale only numeric, passthrough cats
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", scaler, numeric_features),
            ],
            remainder="passthrough"   # leaves categorical columns in place
        )
        
        # 3) build your full inference pipelines
        full_pipe_over = Pipeline([
            ("align",   align_cols),
            ("preproc", preprocessor),
            ("model",   pipe_over)
        ])
        
        full_pipe_under = Pipeline([
            ("align",   align_cols),
            ("preproc", preprocessor),
            ("model",   pipe_under)
        ])
        
        # --- SERIALIZE & UPLOAD TO GRIDFS ---
        for side, pipeline in (('over', full_pipe_over), ('under', full_pipe_under)):
            buf = io.BytesIO()
            joblib.dump(pipeline, buf)
            buf.seek(0)

            fs_id = fs.put(
                buf.read(),
                filename   = f"sbs_nba_ensemble_model_1_{bet_type}_{side}_pipeline_{today_str}.joblib",
                bet_type   = bet_type,
                side       = side,
                created_at = datetime.utcnow()
            )
            print(f"✅ Stored GridFS file id={fs_id} for {bet_type}({side})")

        # collect AUCs for ranking
        avg_auc = (metrics['auc_over'] + metrics['auc_under']) / 2
        results.append({
            'bet_type':  bet_type,
            'auc_over':  metrics['auc_over'],
            'auc_under': metrics['auc_under'],
            'avg_auc':   avg_auc
        })

    # SORT & PRINT RANKING
    results_sorted = sorted(results, key=lambda x: x['avg_auc'], reverse=True)
    print("\n🏆 Bet Type Ranking by Average AUC 🏆")
    for i, r in enumerate(results_sorted, start=1):
        print(f"{i}. {r['bet_type']}: Over AUC={r['auc_over']:.4f}, "
              f"Under AUC={r['auc_under']:.4f}, Avg AUC={r['avg_auc']:.4f}")

    return results_sorted

# USAGE ##############################################
# # get the most recent “over” pipeline for “spread”
# rec = fs.find_one({ 'filename': 'spread_over_pipeline.joblib' }, sort=[('created_at', -1)])
# pipeline_over = joblib.load(io.BytesIO(rec.read()))

# # then
# probs = pipeline_over.predict_proba(X_new)[:, 1]

In [476]:
cache_sbs_nba_ensemble_model_1_trained_pipelines(training_data_by_bet_type)


=== TRAINING player_points ===
Fitting 3 folds for each of 10 candidates, totalling 30 fits
0:	test: 0.5082206	best: 0.5082206 (0)	total: 71ms	remaining: 56.7s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5368471335
bestIteration = 11

Shrink model to first 12 iterations.
0:	test: 0.5032595	best: 0.5032595 (0)	total: 169ms	remaining: 1m 7s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5325438935
bestIteration = 27

Shrink model to first 28 iterations.
0:	test: 0.5211940	best: 0.5211940 (0)	total: 7.65ms	remaining: 6.11s
100:	test: 0.5298547	best: 0.5306551 (77)	total: 570ms	remaining: 3.95s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5306551182
bestIteration = 77

Shrink model to first 78 iterations.
prob_over → best_params = {'clf__learning_rate': 0.001, 'clf__l2_leaf_reg': 1, 'clf__iterations': 800, 'clf__depth': 4}
Fitting 3 folds for each of 10 candidates, totalling 30 fits
0:	test: 0.5116442	best: 0.5116442 (0)	

TypeError: 'NoneType' object is not subscriptable

In [ ]:
# inference

def get_inference_rows():
    today = datetime.today()
    today_str = today.strftime('%Y-%m-%d')
    req = {
        "sports": "BasketballNba",
        "regions": "US",
        "markets": nba_supported_bet_types,
        "oddsFormat": "American",
        "bookmakers": ["DraftKings"]
    }
    events = get_events(req)

    filtered_events = []
    for event in events:
        event_commence_day = event['commenceTime'].split('T')[0]
        if event_commence_day > today or event_commence_day < today:
            filtered_events.append(event)

    for event in filtered_events:
        event_odds_req = req
        event_odds_req['eventId'] = event['id']
        odds = get_event_odds(event_odds_req)
        
        

    

0:	test: 0.5119064	best: 0.5119064 (0)	total: 126ms	remaining: 25s
100:	test: 0.5309497	best: 0.5310270 (85)	total: 18.7s	remaining: 18.3s
199:	test: 0.5322336	best: 0.5324252 (169)	total: 22.1s	remaining: 0us

bestTest = 0.5324251563
bestIteration = 169

Shrink model to first 170 iterations.
0:	test: 0.5082206	best: 0.5082206 (0)	total: 209ms	remaining: 41.7s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5267827489
bestIteration = 14

Shrink model to first 15 iterations.
0:	test: 0.5153273	best: 0.5153273 (0)	total: 184ms	remaining: 2m 27s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5263699878
bestIteration = 44

Shrink model to first 45 iterations.
0:	test: 0.5119064	best: 0.5119064 (0)	total: 206ms	remaining: 1m 22s
100:	test: 0.5331562	best: 0.5334030 (74)	total: 9.87s	remaining: 29.2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5334029868
bestIteration = 74

Shrink model to first 75 iterations.
0:	test: 0.509449